In [ ]:
#!pip install twilio # Install the Twilio library  
# !pip install pandas, requests, beautifulsoup4, tqdm # Install the required libraries for web scraping and data manipulation

In [1]:
import os
from twilio.rest import Client
from twilio_config import TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN, TWILIO_PHONE_NUMBER, API_KEY_WAPI, PHONE_NUMBER_DESTINATION
import time 

from requests import Request, Session
from requests.exceptions import ConnectionError, Timeout, TooManyRedirects
import json
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from datetime import datetime

/Users/mane/Documents/Pipeline_weather/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
city = 'Madrid' # Puedes cambiar la ciudad por la que quieras consultar el clima
api_key = API_KEY_WAPI # Puedes obtener tu API key gratuita en https://www.weatherapi.com/
url = f'http://api.weatherapi.com/v1/forecast.json?key={api_key}&q={city}&days=1&aqi=no&alerts=no' # URL de la API de WeatherAPI para obtener el pronóstico del clima de la ciudad especificada
print(url) # Imprime la URL para verificar que esté correctamente formada
response = requests.get(url).json() # Realiza la solicitud a la API y obtiene la respuesta en formato JSON
print(json.dumps(response, indent=4))  # Cada nivel indentado con 4 espacios

http://api.weatherapi.com/v1/forecast.json?key=6c91bf8820284c8a8a4191341261702&q=Madrid&days=1&aqi=no&alerts=no
{
    "location": {
        "name": "Madrid",
        "region": "Madrid",
        "country": "Spain",
        "lat": 40.4,
        "lon": -3.6833,
        "tz_id": "Europe/Madrid",
        "localtime_epoch": 1771577226,
        "localtime": "2026-02-20 09:47"
    },
    "current": {
        "last_updated_epoch": 1771577100,
        "last_updated": "2026-02-20 09:45",
        "temp_c": 2.3,
        "temp_f": 36.1,
        "is_day": 1,
        "condition": {
            "text": "Sunny",
            "icon": "//cdn.weatherapi.com/weather/64x64/day/113.png",
            "code": 1000
        },
        "wind_mph": 2.2,
        "wind_kph": 3.6,
        "wind_degree": 69,
        "wind_dir": "ENE",
        "pressure_mb": 1029.0,
        "pressure_in": 30.39,
        "precip_mm": 0.0,
        "precip_in": 0.0,
        "humidity": 87,
        "cloud": 0,
        "feelslike_c": 1.7,
   

### Conociendo un poco los datos para poder obtener los valores que me interesan

In [ ]:
response.keys()
# print(json.dumps(response, indent=4))  # ← Cada nivel indentado con 4 espacios
len(response['forecast']['forecastday'][0]['hour'])
# response['forecast']['forecastday'][0]['hour'][1] # Información de la hora 1 del día 0 del pronóstico (primer registro de la lista de horas del primer día del pronóstico)
response['forecast']['forecastday'][0]['hour'][1]['time'].split()[0] # Fecha
int(response['forecast']['forecastday'][0]['hour'][1]['time'].split()[1].split(':')[0]) # Hora
print(json.dumps(response['forecast']['forecastday'], indent=4))  # ← Cada nivel indentado con 4 espacios
response['forecast']['forecastday'][0]['hour'][0]['condition']['text'].strip() # Condición del clima
response['forecast']['forecastday'][0]['hour'][0]['temp_c'] # Temperatura en grados Celsius
response['forecast']['forecastday'][0]['hour'][0]['will_it_rain'] # Probabilidad de lluvia (0 o 1)
response['forecast']['forecastday'][0]['hour'][0]['chance_of_rain'] # Porcentaje de probabilidad de lluvia

## Construimos el Dataframe

### Tenemos 24 registros para cada hora. Ahora, extraeremos cada campo de estos 24 registros de cada hora de weatherapi utilizando una función.

In [3]:
def get_forecast_data(response, i): # Función para extraer los datos de la hora i del día 0 del pronóstico
    
    date = response['forecast']['forecastday'][0]['hour'][i]['time'].split()[0]
    hour = int(response['forecast']['forecastday'][0]['hour'][i]['time'].split()[1].split(':')[0])
    condition = response['forecast']['forecastday'][0]['hour'][i]['condition']['text'].strip()
    temp_c = response['forecast']['forecastday'][0]['hour'][i]['temp_c']
    will_it_rain = response['forecast']['forecastday'][0]['hour'][i]['will_it_rain']
    chance_of_rain = response['forecast']['forecastday'][0]['hour'][i]['chance_of_rain']
    
    return date, hour, condition, temp_c, will_it_rain, chance_of_rain

In [ ]:
datos = []

# tqdm es una biblioteca que muestra una barra de progreso en la consola mientras se ejecuta un bucle, lo que es útil para visualizar el progreso de tareas que pueden llevar tiempo, como la extracción de datos de varias horas del pronóstico del clima. En este caso, se utiliza para iterar sobre cada hora del día 0 del pronóstico y extraer los datos utilizando la función get_forecast_data, mostrando el progreso en la consola.
for i in tqdm(range(len(response['forecast']['forecastday'][0]['hour'])), colour='green'): # Itera sobre cada hora del día 0 del pronóstico y extrae los datos utilizando la función get_forecast_data
    forecast_data = get_forecast_data(response, i)
    datos.append(forecast_data)



100%|██████████| 24/24 [00:00<00:00, 190650.18it/s]


In [ ]:
datos[0][1]

In [ ]:
cols = ['date', 'hour', 'condition', 'temp_c', 'will_it_rain', 'chance_of_rain']
df = pd.DataFrame(datos, columns=cols)
df.head()

In [ ]:
df_rain = df[(df['will_it_rain'] == 1)  & (df['hour'] >= 6) & (df['hour'] <= 20) ]
df_rain = df_rain[['hour', 'condition']]
df_rain.set_index('hour', inplace=True)
if (df_rain.empty):
    print("DF Vacío")

cols_prueba = ['hour', 'condition']
df_rain_prueba = pd.DataFrame(columns=cols_prueba)
df_rain_prueba.loc[len(df_rain_prueba)] = [15, 'Patchy rain nearby']
df_rain_prueba.loc[len(df_rain_prueba)] = [18, 'Patchy rain nearby']
df_rain_prueba

---

## 📤 Enviar Mensaje por Twilio

In [ ]:
# 🧪 PRUEBA: Mensaje ULTRA SIMPLE (sin emojis, solo texto)
# Si el número está verificado pero NO recibes mensajes, puede ser:
# 1. Tu operador bloquea SMS internacionales (USA → España)
# 2. Filtros anti-spam del operador

if df_rain.empty:
    mensaje_sms = "Hoy no va a llover, disfruta del buen clima !"
else: 
    horas_lluvia = ", ".join(df_rain['hour'].astype(str))
    mensaje_sms = f"Alerta Madrid: Lluvia a las {horas_lluvia} horas"

print("🧪 MENSAJE DE PRUEBA (solo texto, sin emojis)")
print("="*50)
print(mensaje_sms)
print("="*50)
print(f"Longitud: {len(mensaje_sms)} caracteres")
print(f"\n🎯 Este mensaje es MUY simple para evitar filtros de spam.")
print(f"💡 Si tampoco llega, el problema es tu operador bloqueando SMS")
print(f"   internacionales de números USA (+1) hacia España (+34).")

In [ ]:
# 🔍 SIMULACIÓN: Cómo se verá el mensaje recibido (con prefijo Trial)

def simular_mensaje_recibido(mensaje, es_trial=True):
    """
    Simula cómo se verá el mensaje en el teléfono del destinatario.
    """
    print("="*60)
    print("📱 SIMULACIÓN DEL MENSAJE RECIBIDO")
    print("="*60)
    
    if es_trial:
        # Twilio agrega este prefijo automáticamente en cuentas trial
        prefijo_trial = "Sent from your Twilio trial account - "
        mensaje_completo = prefijo_trial + mensaje
    else:
        mensaje_completo = mensaje
    
    print(mensaje_completo)
    print("="*60)
    print(f"Longitud total: {len(mensaje_completo)} caracteres")
    print(f"Segmentos SMS: {(len(mensaje_completo) // 160) + 1}")
    print("="*60)

# Simular cómo se verá con cuenta Trial (lo que recibirá el usuario)
print("\n🔴 CON CUENTA TRIAL (como recibiste el mensaje):\n")
simular_mensaje_recibido(mensaje_sms, es_trial=True)

In [ ]:
# 📤 CÓDIGO PARA ENVIAR POR TWILIO
print("🔍 VERIFICACIÓN PREVIA AL ENVÍO")
print(f"📱 Número de origen (Twilio): {TWILIO_PHONE_NUMBER}")
print(f"📱 Número de destino: {PHONE_NUMBER_DESTINATION}")
print(f"📝 Longitud del mensaje: {len(mensaje_sms)} caracteres")
print("\n" + "="*50 + "\n")

In [ ]:
# 🚀 ENVIAR MENSAJE DE PRUEBA SIMPLE
# Ejecuta esta celda para intentar con un mensaje ultra básico

client = Client(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)

message_prueba = client.messages.create(
    body=mensaje_sms,  # Mensaje sin emojis
    from_=TWILIO_PHONE_NUMBER,
    to=PHONE_NUMBER_DESTINATION
)

print("✅ MENSAJE DE PRUEBA ENVIADO")
print("="*50)
print(f"📋 SID: {message_prueba.sid}")
print(f"📊 Estado: {message_prueba.status}")
print(f"\n⏳ Espera 30 segundos y verifica tu móvil...")
print(f"\n❓ Si NO recibes este mensaje simple:")
print(f"   → Tu operador está bloqueando SMS internacionales")
print(f"   → Soluciones: Usar número Twilio español (+34) o WhatsApp API")

In [ ]:
# 📋 VERIFICAR NÚMEROS VERIFICADOS EN CUENTA TRIAL
# Ejecuta esta celda para ver qué números puedes usar

print("📋 NÚMEROS VERIFICADOS EN TU CUENTA TWILIO")
print("="*50)

try:
    # Obtener números verificados (outgoing caller IDs)
    verified_numbers = client.outgoing_caller_ids.list()
    
    if verified_numbers:
        print(f"\n✅ Tienes {len(verified_numbers)} número(s) verificado(s):\n")
        for num in verified_numbers:
            print(f"   📱 {num.phone_number} - {num.friendly_name}")
    else:
        print("\n⚠️  No tienes números verificados.")
        print("   Si tienes cuenta TRIAL, debes verificar números en:")
        print("   https://console.twilio.com/us1/develop/phone-numbers/manage/verified")
    
    # Verificar si el número destino actual está en la lista
    print(f"\n🔍 Verificando número destino: {PHONE_NUMBER_DESTINATION}")
    numeros_verificados_lista = [num.phone_number for num in verified_numbers]
    
    if PHONE_NUMBER_DESTINATION in numeros_verificados_lista:
        print("   ✅ Este número ESTÁ verificado. El mensaje debería llegar.")
    else:
        print("   ❌ Este número NO está verificado.")
        print("   📝 Acción requerida:")
        print("      1. Ve a: https://console.twilio.com/us1/develop/phone-numbers/manage/verified")
        print(f"      2. Verifica el número: {PHONE_NUMBER_DESTINATION}")
        print("      3. Vuelve a enviar el mensaje")
        
except Exception as e:
    print(f"❌ Error al verificar números: {e}")